In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import logging
import os

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class ARIMAModel:
    def __init__(self, p=2, d=1, q=6, P=2, D=1, Q=1, s=24):
        self.p, self.d, self.q = p, d, q
        self.P, self.D, self.Q = P, D, Q
        self.s = s
        self.model = None
        self.fitted = None
        self.last_date = None
        self.offset = 0  # Для сдвига отрицательных значений

    def _check_stationarity(self, series):
        result = adfuller(series)
        logging.info(f'ADF Statistic: {result[0]:.4f}, p-value: {result[1]:.4f}')
        return result[1] < 0.05

    def _plot_acf_pacf(self, series):
        diff = series.diff().dropna() if self.d > 0 else series
        if self.D > 0:
            diff = diff.diff(self.s).dropna()
        
        plt.figure(figsize=(12, 6))
        plt.subplot(121)
        plot_acf(diff, ax=plt.gca())
        plt.title('ACF (дифференцированный ряд)')
        plt.subplot(122)
        plot_pacf(diff, ax=plt.gca())
        plt.title('PACF (дифференцированный ряд)')
        plt.savefig('acf_pacf.png')
        plt.close()

    def _fit_model(self, series):
        # Сдвиг данных для устранения отрицательных значений
        self.offset = -series.min() if series.min() < 0 else 0
        transformed_series = series + self.offset
        self.model = ARIMA(transformed_series, order=(self.p, self.d, self.q), 
                          seasonal_order=(self.P, self.D, self.Q, self.s))
        self.fitted = self.model.fit()
        logging.info(f'Параметры модели: ARIMA({self.p},{self.d},{self.q})x({self.P},{self.D},{self.Q},{self.s})')

    def train(self, data, target='A_plus', group='uuid'):
        if isinstance(data, pd.Series):
            series = data.dropna()
        else:
            series = data[target].dropna()
        
        if len(series) < self.s * 2:
            logging.warning(f'Мало данных, пропуск')
            return
        
        if isinstance(series.index, pd.DatetimeIndex):
            series.index.freq = 'h'
        
        self._plot_acf_pacf(series)
        self._fit_model(series)
        self.last_date = series.index[-1] if isinstance(series.index, pd.DatetimeIndex) else None
        self.check_residuals()

    def forecast(self, steps):
        if self.fitted is None:
            raise ValueError('Модель не обучена')
        forecast = self.fitted.forecast(steps=steps)
        # Обратный сдвиг  
        return forecast - self.offset

    def check_residuals(self):
        if self.fitted is None:
            return
        residuals = self.fitted.resid - self.offset  # Корректировка остатков
        lb_test = acorr_ljungbox(residuals, lags=[1, 12, self.s], return_df=True)
        logging.info(f"Тест Льюнга-Бокса: {lb_test}")
        if lb_test['lb_pvalue'].min() < 0.05:
            logging.warning("Обнаружена автокорреляция в остатках, модель может быть неадекватной")
        plt.figure(figsize=(12, 6))
        plt.subplot(211)
        residuals.plot(title='Остатки')
        plt.subplot(212)
        plot_acf(residuals, lags=self.s, ax=plt.gca())
        plt.title('ACF остатков')
        plt.savefig('residuals.png')
        plt.close()

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def time_series_cv(series, n_splits=2, horizon=24):
    n_train = len(series)
    if n_train < horizon * 2:
        return []   
    
 
    min_train_length = max(48, horizon * 2)  # Должно быть достаточно для модели
    step_size = (n_train - horizon) // n_splits  
    
    if step_size < min_train_length:
        step_size = min_train_length
    
    splits = []
    for i in range(n_splits):
        train_start = i * step_size
        train_end = train_start + step_size
        val_end = train_end + horizon
        
        if val_end > n_train:
            val_end = n_train
            train_end = max(0, val_end - horizon - step_size)
            train_start = max(0, train_end - step_size)
        
        train_data = series[train_start:train_end]
        val_data = series[train_end:val_end]
        
        if len(train_data) >= min_train_length and len(val_data) >= horizon:
            splits.append((train_data, val_data))
    
    return splits

def evaluate_forecast(actual, forecast):
    mae = mean_absolute_error(actual, forecast)
    rmse = np.sqrt(mean_squared_error(actual, forecast))
    return mae, rmse

def predict():
    try:
        file = 'clean_data.csv'
        logging.info(f"Читаю {file}")
        if not os.path.exists(file):
            raise FileNotFoundError(f"Нет файла {file}")
        data = pd.read_csv(file)
    except Exception as e:
        logging.error(f"Ошибка загрузки: {str(e)}")
        raise

    need_cols = ['time_dt', 'A_plus', 'uuid']
    miss_cols = [c for c in need_cols if c not in data.columns]
    if miss_cols:
        raise KeyError(f"Нет столбцов: {miss_cols}. Есть: {data.columns.tolist()}")

    data['time_dt'] = pd.to_datetime(data['time_dt'], errors='raise')
    data = data.set_index('time_dt')

    horizons = [24]
    forecasts = {}
    results = {}

    for uuid in data['uuid'].unique():
        group_data = data[data['uuid'] == uuid]['A_plus']
        if len(group_data) < 48:
            logging.warning(f"Мало данных для uuid={uuid}, пропускаю")
            continue
        
        if not np.isfinite(group_data).all():
            logging.warning(f"Данные для uuid={uuid} содержат NaN или inf")
            continue
        group_data = group_data.resample('h').mean().interpolate(method='linear')
        group_data.index.freq = 'h'

        logging.info(f"uuid={uuid}: Диапазон A_plus: min={group_data.min():.2f}, max={group_data.max():.2f}")

        # Разделение на тренировочную (80%) и тестовую (20%) выборки
        n_train = int(len(group_data) * 0.8)
        train_data = group_data[:n_train]
        test_data = group_data[n_train:]
        logging.info(f"uuid={uuid}: Общая длина данных: {len(group_data)}, тренировочная: {len(train_data)}, тестовая: {len(test_data)}")
 
        if len(train_data) < 72:
            logging.warning(f"Недостаточно данных для кросс-валидации uuid={uuid}, пропускаю кросс-валидацию")
            cv_maes = []
        else:
  
            logging.info(f"Кросс-валидация для uuid={uuid}")
            cv_splits = time_series_cv(train_data, n_splits=2, horizon=max(horizons))
            cv_maes = []

            for train_split, val_split in cv_splits:
                logging.info(f"Длина train_split: {len(train_split)}, val_split: {len(val_split)}")
                if len(train_split) < 24 or len(val_split) < max(horizons):
                    logging.warning(f"Недостаточно данных для фолда, пропускаю")
                    continue
                try:
                    arima = ARIMAModel(p=2, d=1, q=6, P=2, D=1, Q=1, s=24)
                    arima.train(train_split)
                    forecast = arima.forecast(steps=len(val_split))
                    if len(forecast) < len(val_split):
                        raise ValueError("Недостаточная длина прогноза")
                    mae, _ = evaluate_forecast(val_split, forecast)
                    cv_maes.append(mae)
                except Exception as e:
                    logging.error(f"Ошибка в кросс-валидации для uuid={uuid}: {str(e)}")
                    continue

            if cv_maes:
                logging.info(f"Среднее MAE на кросс-валидации для uuid={uuid}: {np.mean(cv_maes):.2f}")
            else:
                logging.warning(f"Кросс-валидация для uuid={uuid} не выполнена")

        logging.info(f"Обучаю модель для uuid={uuid}")
        arima = ARIMAModel(p=2, d=1, q=6, P=2, D=1, Q=1, s=24)
        try:
            arima.train(train_data)
        except Exception as e:
            logging.error(f"Ошибка обучения модели для uuid={uuid}: {str(e)}")
            continue

       
        for horizon in horizons:
            forecast = arima.forecast(steps=horizon)
            actual = test_data[:horizon]
            mae, rmse = evaluate_forecast(actual, forecast)
            logging.info(f"uuid={uuid}, горизонт={horizon}: MAE={mae:.2f}, RMSE={rmse:.2f}")
            forecasts[(uuid, horizon)] = forecast
            results[(uuid, horizon)] = (mae, rmse)
 
        forecast_24 = arima.forecast(steps=24)
        forecasts[uuid] = forecast_24
        logging.info(f"Прогноз для uuid={uuid}:\n{forecast_24}")

    try:
        forecast_items = [pd.DataFrame({'uuid': uuid, 'time_dt': f.index, 'forecast': f.values}) 
                         for uuid, f in forecasts.items() if not isinstance(uuid, tuple)]
        if forecast_items:
            forecasts_df = pd.concat(forecast_items, ignore_index=True)
            forecasts_df.to_csv('forecasts.csv', index=False)
            logging.info("Прогнозы сохранены в forecasts.csv")
        else:
            logging.warning("Нет прогнозов для сохранения")
    except Exception as e:
        logging.error(f"Ошибка сохранения: {str(e)}")
        raise

if __name__ == "__main__":
    predict()

2025-05-11 18:01:34,112 - INFO - Читаю clean_data.csv
2025-05-11 18:01:34,232 - INFO - uuid=03014b53-9900-4d9a-b78c-2959d439a8eb: Диапазон A_plus: min=-0.90, max=13.29
2025-05-11 18:01:34,233 - INFO - uuid=03014b53-9900-4d9a-b78c-2959d439a8eb: Общая длина данных: 9597, тренировочная: 7677, тестовая: 1920
2025-05-11 18:01:34,233 - INFO - Кросс-валидация для uuid=03014b53-9900-4d9a-b78c-2959d439a8eb
2025-05-11 18:01:34,233 - INFO - Длина train_split: 3826, val_split: 24
2025-05-11 18:06:01,204 - INFO - Параметры модели: ARIMA(2,1,6)x(2,1,1,24)
2025-05-11 18:06:01,224 - INFO - Тест Льюнга-Бокса:       lb_stat  lb_pvalue
1    0.049326   0.824239
12   7.783483   0.801813
24  11.344036   0.986340
2025-05-11 18:06:01,475 - INFO - Длина train_split: 3826, val_split: 24
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Li